### Imports

In [ ]:
import os
import json
from pypdf import PdfReader
from tqdm import tqdm

### Path

In [ ]:
DATA_ROOT = "../../.."
OUTPUT_PATH = "../data/processed/parsed_data.json"

### Collecting all files

In [ ]:
def collect_files(root_dir):
    """
    Recursively walks through a directory
    and collects all file paths.
    """
    all_files = []
    
    for root, _, files in os.walk(root_dir):
    
        for file in files:
            full_path = os.path.join(root, file)
            all_files.append(full_path)
    
    return all_files


files = collect_files(DATA_ROOT)

print(f"Total files found: {len(files)}")
print(os.listdir(DATA_ROOT))

### Detect File Type

In [ ]:
def get_file_type(path):
    """
    Detect file type based on extension
    """
    if path.endswith(".md"):
        return "md"
    elif path.endswith(".ipynb"):
        return "ipynb"
    elif path.endswith(".py"):
        return "py"
    elif path.endswith(".pdf"):
        return "pdf"
    else:
        return "other"

### Parse Markdown Files

In [ ]:
def parse_md(path):
    """
    Reads markdown file as plain text
    """
    try:
        with open(path, "r", encoding="utf-8") as f:
            return f.read()
    except Exception as e:
        print(f"Error reading {path}: {e}")
        return ""

### Parse Jupyter Notebooks

In [ ]:
def parse_ipynb(path):
    """
    Parses a Jupyter notebook and returns structured cells
    
    IMPORTANT:
    - We do NOT merge markdown and code here
    - We preserve the original structure
    """
    try:
        with open(path, "r", encoding="utf-8") as f:
            notebook = json.load(f)
        
        cells = []
        
        for cell in notebook.get("cells", []):
            cell_type = cell.get("cell_type")
            source = "".join(cell.get("source", []))
            
            cells.append({
                "type": cell_type,
                "text": source
            })
        
        return cells
    
    except Exception as e:
        print(f"Error parsing ipynb {path}: {e}")
        return []

### Parse Python Files

In [ ]:
def parse_py(path):
    """
    Reads Python file as raw text
    
    IMPORTANT:
    No semantic analysis here — just extraction
    """
    try:
        with open(path, "r", encoding="utf-8") as f:
            return f.read()
    except Exception as e:
        print(f"Error reading {path}: {e}")
        return ""

### Parse PDF Files

In [ ]:
def parse_pdf(path):
    """
    Extracts text from PDF files
    """
    try:
        reader = PdfReader(path)
        text = ""
        
        for page in reader.pages:
            text += page.extract_text() or ""
        
        return text
    
    except Exception as e:
        print(f"Error parsing PDF {path}: {e}")
        return ""

### Detect Project Name (Metadata)

In [ ]:
def detect_project(path):
    """
    Detects project name based on file path
    
    Example:
    data/raw/churn/... → churn
    """
    parts = path.split(os.sep)
    
    if "churn" in parts:
        return "churn"
    elif "traffic" in parts:
        return "traffic_sign"
    elif 'Courses' in parts:
        return 'courses'
    elif 'scientific publications' in parts:
        return 'scientific_publications'
    else:
        return "unknown"

### Main Parsing Pipeline

In [ ]:
parsed_data = []

for path in tqdm(files):
    
    file_type = get_file_type(path)
    
    # skip irrelevant files
    if file_type == "other":
        continue
    
    project = detect_project(path)
    
    if file_type == "md":
        content = parse_md(path)
    
    elif file_type == "ipynb":
        content = parse_ipynb(path)
    
    elif file_type == "py":
        content = parse_py(path)
    
    elif file_type == "pdf":
        content = parse_pdf(path)
    
    else:
        continue
    
    parsed_data.append({
        "file": path,
        "project": project,
        "type": file_type,
        "content": content
    })

In [ ]:
len(parsed_data)

In [ ]:
parsed_data[0]

### Save to JSON

In [ ]:
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(parsed_data, f, ensure_ascii=False, indent=2)

print(f"Saved to {OUTPUT_PATH}")

### Summary

- Implemented a pipeline to **collect and process files** from the portfolio directory  
- Supported multiple formats: `.md`, `.ipynb`, `.py`, `.pdf`  
- Extracted raw content while preserving structure (especially for notebooks)  
- Added basic metadata (project name, file type, path)  
- Saved all parsed data into a **structured JSON file**  

Result: unified raw dataset ready for semantic processing (next step in RAG pipeline)